# Sprint 9 · Webinar 29 · Data Analytics práctico  
**Student Version · Proyecto Sprint 9: Experimento A/B en página de inicio (Landing Experiment)**

> En esta sesión práctica trabajarás el proyecto **"Landing Experiment"** paso a paso.  
> La idea no es solo ejecutar código, sino **pensar, documentar, interpretar y decidir**.

**Usa este notebook para:**
- tomar apuntes,
- completar código,
- registrar resultados,
- interpretar métricas,
- justificar decisiones de negocio.

> Cuando veas `TODO`, significa que debes completar esa parte durante la clase.


## Fecha
> Completa aquí la fecha de la sesión.


## Objetivo de la sesión práctica (100 min)

Al finalizar esta clase, podrás:

1. Cargar y validar un dataset de un **experimento A/B**.
2. Realizar un **EDA básico** para detectar problemas de calidad (tipos, duplicados, valores fuera de rango).
3. Comparar **métricas clave** entre las versiones **A** y **B**:
   - **Gasto promedio** (solo usuarios que convirtieron)
   - **Tasa de conversión** (usuarios que convierten / total)
4. Aplicar pruebas simples para tomar una decisión:
   - **t-test** para comparar promedios
   - **Chi-cuadrado** para comparar proporciones (conversiones)
5. Escribir conclusiones claras: **qué pasó, qué significa, y qué haría negocio**.


## Agenda sugerida (100 minutos)

1. Contexto del proyecto + checklist de entregables (5 min)  
2. **Actividad 0 (breakout rooms):** de pregunta de negocio a métrica (10 min)  
3. Ejercicio 1: cargar y entender el dataset (15 min)  
4. Ejercicio 2: revisión de calidad de datos (15 min)  
5. Ejercicio 3: gasto promedio (A vs B) + visualización (15 min)  
6. Ejercicio 4: prueba t para el gasto (15 min)  
7. Ejercicio 5: tasa de conversión (A vs B) + Chi-cuadrado (15 min)  
8. Ejercicio 6: segmentación rápida (región / dispositivo) (5 min)  
9. Reporte final + cierre (5 min)


## Contexto del negocio

Un equipo de producto probó dos versiones de la **página de inicio (landing page)**:

- **Versión A:** diseño actual  
- **Versión B:** nuevo diseño (hipótesis: mejora conversión y/o gasto)

El objetivo del experimento es responder preguntas como:

- ¿La versión B **convierte más** usuarios?
- Entre quienes convierten, ¿la versión B genera **más gasto**?

⚠️ Importante:
- **Conversión** suele ser una variable binaria (`0/1`).
- **Gasto** es numérica y suele tener mucha variabilidad.


## Dataset del proyecto

En el proyecto trabajaremos con un archivo tipo CSV llamado `landing_experiment_sintetico_40k.csv`.

Columnas típicas (pueden variar ligeramente según tu versión):
- `user_id` — identificador único del usuario  
- `date` — fecha en que el usuario participó en el experimento  
- `landing` — variante del experimento (`A` o `B`)  
- `converted` — si convirtió (1) o no (0)  
- `gasto` — gasto del usuario (normalmente 0 si no convirtió, o gasto positivo si convirtió)  
- Otras variables de segmentación: `region`, `dispositivo`, `traffic_source`, `user_type`, etc.

📌 **Checklist de entregables (lo que se espera en el proyecto):**
- Validación básica de datos (tipos, duplicados, rango de fechas)
- Comparación de métricas A vs B
- Pruebas estadísticas simples + interpretación
- Conclusión de negocio y recomendación


---

# Actividad 0 · Calentamiento (10 min)

Tus estudiantes aún están iniciando, así que **no empezamos** con fórmulas.

### Instrucciones (breakout rooms, 3–4 personas)

1) Lean la pregunta:  
   **“¿La versión B es mejor que la A?”**

2) Conviertan la pregunta en **2 preguntas medibles** (métrica + comparación).  
   Usa esta plantilla:

- Métrica 1: ___________  
  Comparación: A vs B  
  “B es mejor si _________”

- Métrica 2: ___________  
  Comparación: A vs B  
  “B es mejor si _________”

3) Definan qué columnas del dataset usarían para cada métrica.

### Pista
En experimentos A/B casi siempre aparecen:
- una métrica de **conversión** (proporción)
- una métrica de **valor** (promedio/mediana de gasto)

### Puesta en común (2 min)
Cada equipo comparte sus 2 métricas.


In [ ]:
# ============================================================
# Imports y configuración (ejecuta esta celda primero)
# ============================================================
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import chi2_contingency

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

---

# Ejercicio 1 · Cargar y explorar el dataset (15 min)

**Meta:** cargar datos, ver estructura y entender qué representa cada columna.

### 1.1 Cargar datos

Usaremos el archivo `landing_experiment_sintetico_40k.csv`. Disponible en GitHub:


In [ ]:
# Tu turno: carga el dataset desde GitHub
# Pista: usa pd.read_csv(...)

url = "https://raw.githubusercontent.com/ljpiere/tpdata_python/refs/heads/main/DA/datasets/landing_experiment_sintetico_40k.csv"

# TODO: carga el dataset en una variable llamada df
# df = ...

# TODO: muestra las primeras filas
# df.head()

In [ ]:
# Vista general: dimensiones y tipos

# TODO: imprime las dimensiones del dataset
# print("Dimensiones (filas, columnas):", ...)

# TODO: revisa la estructura general
# df.info()

✍️ **Mini-análisis (texto en Markdown):**
- ¿Cuántas filas y columnas hay?
- ¿Qué columnas parecen ser numéricas? ¿cuáles categóricas?
- ¿Qué columna representa la variante (A/B)? ¿y la conversión?

> Escribe tus observaciones debajo de este bloque.


### 1.2 Diccionario de datos (para la sesión)

Completa este diccionario con lo que observes:

- `user_id`: __________  
- `date`: __________  
- `landing`: __________  
- `converted`: __________  
- `gasto`: __________  
- otras columnas: __________


---

# Ejercicio 2 · Revisión de calidad de datos (15 min)

**Meta:** asegurar que el experimento está bien formado.

Checklist:
1. ¿Hay valores faltantes?
2. ¿Hay `user_id` duplicados?
3. ¿`landing` solo tiene A y B?
4. ¿`converted` solo tiene 0 y 1?
5. ¿`gasto` tiene valores negativos? (no debería)


In [ ]:
# TODO: revisa valores faltantes por columna
# Escribe una línea que calcule los nulos por columna

# df. ...

In [ ]:
# 2.1 Porcentaje de faltantes por columna

# TODO: calcula el porcentaje de valores faltantes y ordénalo de mayor a menor
# missing_pct = ...

# TODO: muéstralo como DataFrame y redondea a 2 decimales
# ...

In [ ]:
# 2.2 Duplicados en user_id
# Si cada usuario aparece una sola vez, no debería haber duplicados.

# TODO: calcula cuántos user_id duplicados existen
# duplicados_user = ...

# TODO: imprime el resultado
# print("Duplicados en user_id:", duplicados_user)

In [ ]:
# 2.3 Valores únicos en variables clave

# TODO: revisa los valores distintos de landing
# print("Valores en landing:", ...)

# TODO: revisa los valores distintos de converted
# print("Valores en converted:", ...)

In [ ]:
# Espacio opcional de validación adicional
# Usa esta celda si quieres probar una segunda forma de revisar valores únicos.

# Ejemplo:
# sorted(df["landing"].dropna().unique())

In [ ]:
# Revisión rápida de gasto

# TODO: imprime el valor mínimo de gasto
# print("Gasto mínimo:", ...)

# TODO: imprime el valor máximo de gasto
# print("Gasto máximo:", ...)

# TODO: muestra estadísticas descriptivas
# ...

### 2.4 Fecha: rango temporal del experimento (opcional pero recomendado)

Un experimento A/B normalmente tiene un rango de fechas definido.
Revisemos mínimos y máximos.


In [ ]:
# Convierte date a datetime y revisa el rango temporal del experimento

# TODO: convierte la columna date usando errors="coerce"
# df["date"] = ...

# TODO: imprime fecha mínima y fecha máxima
# print("Fecha mínima:", ...)
# print("Fecha máxima:", ...)

In [ ]:
# ¿Hay fechas inválidas?

# TODO: cuenta cuántas fechas quedaron como NaT
# print("Fechas inválidas (NaT):", ...)

✍️ **Decisiones (documentación):**
Si encontraste problemas, documenta qué harías. Ejemplos:

- Si hay duplicados en `user_id`: ¿los eliminas? ¿por qué?
- Si hay gasto negativo: ¿filtras? ¿lo marcas como error?
- Si `date` tiene NaT: ¿remueves esas filas?

> Nota: En esta clase **no** buscamos “perfección”, buscamos hábitos sanos: revisar y documentar.


### 2.5 Limpieza mínima (para poder analizar)

Aplicaremos una limpieza **muy simple** y conservadora:

- Remover filas con `landing` faltante
- Asegurar `converted` como entero 0/1 (si es posible)
- Reemplazar `gasto` faltante por 0 (si existe), porque suele significar “no gastó”

⚠️ Si tu dataset real es distinto, ajusta con criterio y documenta.


In [ ]:
# Copia de trabajo para no modificar df directamente
# TODO: crea una copia llamada data
# data = ...

# 1) Eliminar filas sin landing
# TODO: aplica una limpieza mínima
# data = ...

In [ ]:
# 2) Asegurar converted como variable numérica 0/1 cuando sea posible

# TODO: convierte converted a numérico usando errors="coerce"
# data["converted"] = ...

In [ ]:
# 3) Gasto: convertir a numérico y completar faltantes con 0

# TODO: convierte gasto a numérico
# data["gasto"] = ...

# TODO: rellena faltantes de gasto con 0
# data["gasto"] = ...

# TODO: imprime las dimensiones después de la limpieza mínima
# print("Dimensiones después de limpieza mínima:", ...)

# TODO: visualiza algunas columnas clave
# data[[...]].head()

---

# Ejercicio 3 · Gasto promedio (A vs B) (15 min)

**Pregunta:** Entre quienes convirtieron (`converted = 1`), ¿el gasto promedio es diferente en A vs B?

### Paso 1: filtrar solo convertidos
### Paso 2: calcular promedio y tamaño de muestra (n)
### Paso 3: visualizar (histograma simple o boxplot)


In [ ]:
# 3.1 Filtrar solo usuarios que convirtieron

# TODO: crea un DataFrame conv solo con converted == 1
# conv = ...

# TODO: imprime cuántos usuarios convirtieron
# print("Usuarios que convirtieron:", ...)

# TODO: revisa cuántos hay por variante
# print(...)
# conv.head()

In [ ]:
# 3.2 Resumen por variante: tamaño (n) y gasto promedio

# TODO: agrupa por landing y calcula:
# - count
# - mean
# - median
# - std
# resumen_gasto = ...

# TODO: renombra columnas para que queden más claras
# ...

In [ ]:
# 3.3 Visualización simple del gasto por variante
# Puedes usar boxplot o histograma.

# TODO: crea una visualización para comparar gasto entre A y B
# Pista: usa conv.boxplot(...) o plt.hist(...)

# plt.figure()
# ...
# plt.show()

✍️ **Interpretación visual:**
- ¿Ves una diferencia clara entre A y B?
- ¿Hay muchos valores extremos (outliers)?
- ¿La distribución parece “rara” (muy sesgada)?

> Escribe 2–3 frases.


---

# Ejercicio 4 · Prueba t para comparar gasto promedio (15 min)

Ahora pasamos de “ver diferencias” a **validar con una prueba estadística simple**.

### ¿Qué estamos comparando?
- Dos grupos independientes (usuarios que vieron A vs usuarios que vieron B)
- Variable numérica: `gasto`
- Solo usuarios que convirtieron (porque quienes no convierten suelen tener gasto 0)

### Prueba elegida: t-test (Welch)
Usamos `ttest_ind(..., equal_var=False)` porque **no asumimos varianzas iguales** (es más seguro).

📌 Interpretación (regla simple):
- Si `p_value < 0.05`: evidencia de diferencia (al nivel 5%)
- Si `p_value >= 0.05`: no hay evidencia suficiente para afirmar diferencia

⚠️ Esto no significa “B es mejor” automáticamente: hay que mirar también el **tamaño** de la diferencia (promedios).


In [ ]:
# 4.1 Separar gasto por variante (solo convertidos)

# TODO: crea dos Series:
# gasto_A = ...
# gasto_B = ...

# TODO: imprime n y promedio de cada grupo
# print(...)
# print(...)

In [ ]:
# 4.2 t-test de Welch (varianzas no asumidas iguales)

# TODO: aplica stats.ttest_ind(..., equal_var=False)
# t_stat, p_value = ...

# TODO: imprime estadístico t y p-value
# print("t_stat:", ...)
# print("p_value:", ...)

### 4.3 Conclusión en lenguaje de negocio

Completa:

- Promedio A: ______  
- Promedio B: ______  
- p-value: ______

**Conclusión (1–2 frases):**  
“Con un nivel de significancia del 5%, _________ evidencia de que el gasto promedio entre convertidos sea diferente entre A y B.  
En términos prácticos, B tiene un gasto promedio _________ que A (mayor/menor/similar).”


---

# Ejercicio 5 · Tasa de conversión (A vs B) + Chi-cuadrado (15 min)

**Pregunta:** ¿La versión B convierte una proporción distinta de usuarios que la A?

### 5.1 Tasa de conversión
Definición:
\[
	ext{conversion\_rate} = \frac{\#converted=1}{\#total}
\]

### 5.2 Prueba estadística: Chi-cuadrado (2×2)

Construimos una tabla de contingencia:

| landing | converted=0 | converted=1 |
|---|---:|---:|
| A | ... | ... |
| B | ... | ... |

Luego aplicamos `chi2_contingency`.  
Regla simple:
- `p_value < 0.05` → evidencia de que la conversión depende de la variante (A/B)


In [ ]:
# 5.1 Primera revisión rápida
# TODO: calcula la tasa de conversión promedio por landing
# Pista: converted suele estar codificada como 0/1

# ...

In [ ]:
# TODO: revisa cuántas observaciones hay por variante
# Esto te ayuda a validar tamaños de muestra

# ...

In [ ]:
# 5.1 Tasa de conversión por variante

# TODO: calcula conversion rate por landing
# conv_rate = ...

# TODO: calcula cantidad total por variante
# counts = ...

In [ ]:
# TODO: construye un DataFrame resumen con:
# - n_total
# - conversion_rate

# resumen_conv = ...
# resumen_conv

In [ ]:
# 5.2 Tabla de contingencia 2x2 (landing vs converted)

# TODO: crea una tabla de contingencia
# tabla = ...

# tabla

In [ ]:
# 5.3 Prueba Chi-cuadrado

# TODO: aplica chi2_contingency(tabla)
# chi2, p_value, dof, expected = ...

# TODO: imprime chi2, p_value y dof
# print(...)
# print(...)
# print(...)

# TODO opcional: convierte expected en DataFrame para revisarlo
# expected_df = ...
# expected_df

### 5.4 Conclusión de conversión (lenguaje de negocio)

Completa:

- Conversion rate A: ______  
- Conversion rate B: ______  
- p-value: ______

**Conclusión (1–2 frases):**  
“Con un nivel de significancia del 5%, _________ evidencia de que la tasa de conversión sea diferente entre A y B.  
En términos prácticos, B convierte _________ que A (más/menos/similar).”


---

# Ejercicio 6 · Segmentación rápida (5 min)

A veces, el efecto cambia por segmento (región, dispositivo, fuente de tráfico).

**Meta:** calcular la tasa de conversión por `landing` dentro de **un** segmento.

Elige una columna de segmentación que exista en tu dataset, por ejemplo:
- `region`
- `dispositivo`
- `traffic_source`
- `user_type`

Si tu dataset no tiene esas columnas, puedes saltar este ejercicio.


In [ ]:
# Tu turno: revisa qué columnas de segmentación existen

# TODO: muestra el listado de columnas
# data.columns

In [ ]:
# Segmentación rápida
# Elige una columna de segmentación que sí exista en tu dataset.

segment_col = "dispositivo"  # TODO: cambia este valor si no existe en tu dataset

# TODO:
# 1) valida si la columna existe
# 2) calcula conversion rate por segmento y landing
# 3) reestructura el resultado para comparar A vs B
# 4) ordénalo para detectar dónde parece funcionar mejor B

# Pista general:
# if segment_col in data.columns:
#     seg = ...
#     seg
# else:
#     print(...)

✍️ **Mini-insight (1 frase):**
- ¿En qué segmento parece que B mejora más (si es que mejora)?


---

# Reporte final (5 min)

Completa este mini-reporte. La idea es practicar cómo comunicar resultados.

## 1) ¿Qué medimos?
- Métrica 1 (conversión): _________  
- Métrica 2 (gasto en convertidos): _________  

## 2) Resultados principales
- Conversión A vs B: _________  
- Gasto promedio (convertidos) A vs B: _________  

## 3) Evidencia estadística (regla simple con p-value)
- p-value conversión (Chi-cuadrado): _________  
- p-value gasto (t-test): _________  

## 4) Conclusión de negocio (decisión)
Elige una:
- ✅ Recomendar B
- ❌ Mantener A
- 🟡 No hay evidencia suficiente → recolectar más datos / extender experimento

Justifica en 2–3 frases: _________

## 5) Limitaciones (menciona 1–2)
Ejemplos: duración del experimento, sesgo por fuente de tráfico, estacionalidad, usuarios repetidos, etc.


## Takeaways de la sesión práctica

- Un A/B test se resuelve con un flujo ordenado: **cargar → validar → medir → comparar → decidir**.
- No basta con “ver diferencias”: usamos pruebas simples para evaluar si la diferencia podría ser azar.
- La decisión final debe considerar:
  - **tamaño del efecto** (diferencia real en métricas)
  - **evidencia estadística** (p-value)
  - **contexto del negocio** (costos, riesgos, impacto)


### ¿Necesitas ayuda?
Recuerda los canales de ayuda que tenemos disponibles:
- `DATA-CO-LEARNING`: Revisa los horarios de atencion en la guia dele studiante. Recuerda que tenemos horarios de apoyo todos los días para tus dudas puntuales!
- `DA_CONSULTA`: Uuedes publicar tus preguntas sobre el contenido de la plataforma o el proyecto 24/7. Recuerda que en tu pregunta debes de agregar el tag @Dataconsulta.
- `SPRINT FOCUS`: Para los Sprints 1 al 5 tenemos sesiones extra donde abordamos temáticas de cada sprint a rpofundidad. Revisa la guia del estudiante para conocer los horarios!
- `SESIONES 1:1`: Recuerda que puedes agendar sesiones 1:1 con un tutor. Agendalas con antelación y resuelve todas tus dudas, desde temás del proyecto, curso, carrera hasta problemas técnicos.
- `CANAL DE DISCORD DE COMMUNITY`: Recuerda que tu cohorte tiene un canal especial donde puedes compartir cualquier item interesante que quieras mostrar a tus compañeros de curso.

## Cierre y próximos pasos

Para el entregable del proyecto:
- Mantén tu notebook **limpio** (Markdown + resultados).
- Documenta decisiones de limpieza y supuestos.
- Termina con una **recomendación clara** (y limitaciones).

Siguiente nivel (para más adelante):
- intervalos de confianza para métricas
- pruebas no paramétricas si hay distribuciones muy sesgadas
- métricas adicionales (retención, click-through, etc.)

Ayuda en camino:
- Recuerda que puedes agendar sesiones 1:1 para revisar futuros entregables de tu proyecto.
- DA_CONSULTAS abierto para tus preguntas sobre el proeycto.
- DATA_CO_LEARNING disponible para preguntas rápida.


In [ ]:
# Espacio libre de trabajo
# Usa esta celda para:
# - pruebas extra,
# - validaciones,
# - gráficos opcionales,
# - borradores para el reporte final.
